# 5.主动退出

主动退出 = 在循环执行过程中主动终止执行。LangGraph 有两种手段，推荐**分工使用**：

- **限制类退出（步数、工具上限）-> 条件边 route**：route 是只读 state 的纯函数，出口全部集中在 `add_conditional_edges`（本节案例）
- **运行时突发事件（LLM 抛异常、超时、interrupt）-> 节点内 `Command(goto=END)`**：决策依赖执行过程信息，纯函数 route 表达不了

案例组合三层限制：`MAX_STEPS`（LLM 调用总步数）、`MAX_TOTAL_TOOLS`（工具调用总量）、
`MAX_PER_TOOL`（单项限制）。计数器放进 State 用增量 reducer，`over_limits` 是所有限制判断的共享纯函数。


In [ ]:
# 代码案例：主动退出 —— 限制类退出走「条件边 + 共享纯函数」，Command 只留给运行时突发事件
#
# 设计分工（推荐模式）：
# - 限制类判断（步数 / 工具上限）→ route 纯函数挂条件边：
#   全图出口 = 两条 add_conditional_edges，拓扑一目了然，route 可脱离图单测
# - 运行时突发事件（LLM API 抛异常）→ 节点内 Command(goto=END)：
#   决策依赖执行过程信息，「节点跑完后调纯函数」的 route 表达不了这种信号
#
# 三层限制：MAX_STEPS（推理轮次）、MAX_TOTAL_TOOLS（动作总量）、MAX_PER_TOOL（单项限制）
# 硬兜底：recursion_limit（invoke config，按节点执行数计，一环 = 两次执行）

import subprocess
from operator import add
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command
from rich import print as rprint
from tavily import TavilyClient

load_dotenv()

_tavily = TavilyClient()
model = ChatDeepSeek(model="deepseek-v4-flash")


# ---- 工具定义 ----


@tool(parse_docstring=True)
def read(file_path: str) -> str:
    r"""读取本地文本文件的完整内容，以 UTF-8 编码返回。

    适合读取代码、配置、文档等文本文件。不支持二进制文件（如图片、Excel）。
    文件不存在或读取失败时，返回以"读取失败:"开头的错误信息，而非抛出异常。

    Args:
        file_path: 文件路径，支持绝对路径（如 D:/Work/project/main.py）
            或相对于工作目录的相对路径（如 src/config.json）。分隔符用 / 或 \ 均可。

    Returns:
        文件全文内容；失败时返回"读取失败: <原因>"。
    """
    try:
        return Path(file_path).read_text(encoding="utf-8")
    except Exception as e:
        return f"读取失败: {e}"


@tool(parse_docstring=True)
def write(file_path: str, content: str) -> str:
    """将文本内容写入本地文件（UTF-8 编码），会覆盖文件原有内容。

    父目录不存在时会自动创建。适合保存代码、配置、笔记等文本文件。

    Args:
        file_path: 目标文件路径，支持绝对或相对路径，如 D:/Work/demo/output.txt。
        content: 要写入的完整文本内容，为空字符串时会清空文件。

    Returns:
        成功返回"写入成功: <文件路径>"；失败返回"写入失败: <原因>"。
    """
    try:
        Path(file_path).parent.mkdir(parents=True, exist_ok=True)
        Path(file_path).write_text(content, encoding="utf-8")
        return f"写入成功: {file_path}"
    except Exception as e:
        return f"写入失败: {e}"


@tool(parse_docstring=True)
def shell(command: str) -> str:
    """在 Windows 上执行 PowerShell 命令并返回输出结果，可运行任何命令行操作。

    例如：查看目录（Get-ChildItem）、运行 Python 脚本（python main.py）、
    安装依赖（pip install requests）、Git 操作（git status）等。
    命令超时时间为 30 秒，长时间运行的命令（如训练模型）请勿使用此工具。
    路径中的反斜杠会自动转换为正斜杠，无需额外处理。

    Args:
        command: 要执行的 PowerShell 命令，如 "Get-ChildItem D:/Work" 或
            "python -m pytest tests/ -v"。多个命令用 ; 分隔。

    Returns:
        成功时返回命令的标准输出（stdout），无输出时返回"(无输出)"；
        失败返回"错误: <stderr>"；超过 30 秒返回"命令执行超时"。
    """
    try:
        result = subprocess.run(
            ["powershell", "-Command", command],
            capture_output=True,
            text=True,
            timeout=30,
        )
        if result.returncode == 0:
            return result.stdout.strip() or "(无输出)"
        error = result.stderr.strip() or result.stdout.strip() or "未知错误"
        return f"错误: {error}"
    except subprocess.TimeoutExpired:
        return "命令执行超时"
    except Exception as e:
        return f"执行失败: {e}"


@tool(parse_docstring=True)
def web_search(query: str, max_results: int = 5) -> str:
    r"""联网搜索最新信息，返回与查询相关的网页标题、链接和内容摘要。

    适合查询实时或模型不知道的信息（新闻、最新版本、价格、文档等）。
    搜索失败时返回以"搜索失败:"开头的错误信息，而非抛出异常。

    Args:
        query: 搜索关键词或自然语言问题，如 "LangGraph 最新版本"。
        max_results: 返回结果数量，默认 5，建议 3~10。

    Returns:
        每条结果包含标题、链接和内容摘要的文本；失败时返回"搜索失败: <原因>"。
    """
    try:
        max_results = max(1, min(max_results, 10))
        res = _tavily.search(
            query=query, max_results=max_results, include_answer="basic"
        )
        parts = []
        if res.get("answer"):
            parts.append(f"参考答案: {res['answer']}")
        for r in res.get("results", []):
            parts.append(f"- {r['title']}\n  {r['url']}\n  {r['content']}")
        return "\n\n".join(parts) or "未找到相关结果"
    except Exception as e:  # noqa: BLE001
        return f"搜索失败: {e}"


# ---- 绑定工具 ----
tools = [read, write, shell, web_search]
model_with_tools = model.bind_tools(tools)

# ---- 三层限制的上限配置 ----
MAX_STEPS = 8  # 推理轮次上限
MAX_TOTAL_TOOLS = 12  # 动作总量上限
MAX_PER_TOOL = {"web_search": 3, "write": 3, "shell": 5}  # 单项工具上限


# ---- 计数器 reducer：工具名 -> 次数，按 key 累加（增量语义）----
def add_counts(a: dict[str, int], b: dict[str, int]) -> dict[str, int]:
    out = dict(a or {})
    for k, v in (b or {}).items():
        out[k] = out.get(k, 0) + v
    return out


class ChatState(MessagesState):
    steps: Annotated[int, add]  # 节点返回增量 {"steps": 1}
    tool_counts: Annotated[dict[str, int], add_counts]  # 节点返回增量 {"web_search": 1}


def over_limits(state: ChatState) -> bool:
    """工具调用是否已达上限（总量或任一单项）—— 纯函数：只读 state，可脱离图直接测试"""
    counts = state.get("tool_counts", {})
    if sum(counts.values()) >= MAX_TOTAL_TOOLS:
        print("工具调用已达最大次数")
        return True
    return any(counts.get(name, 0) >= limit for name, limit in MAX_PER_TOOL.items())


def llm_node(state: ChatState):
    try:
        ai_msg = model_with_tools.invoke(state["messages"])
    except Exception as e:
        rprint(f"LLM 调用异常，主动退出: {e}")
        return Command(
            goto=END
        )  # 运行时突发事件：错误不在 state 里，纯函数 route 表达不了
    if ai_msg.tool_calls:
        return {"messages": [ai_msg], "steps": 1}
    return {"messages": [ai_msg], "steps": 1}


def tool_node(state: ChatState) -> dict:
    last = state["messages"][-1]
    tool_map = {tool_.name: tool_ for tool_ in tools}
    tool_messages = []
    count_delta = {}
    for call in last.tool_calls:  # type: ignore
        selected = tool_map.get(call["name"])
        if selected is None:
            result = f"工具调用失败: 未知工具 {call['name']}"
        else:
            result = selected.invoke(call["args"])
        tool_messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
        count_delta[call["name"]] = count_delta.get(call["name"], 0) + 1
    return {"messages": tool_messages, "tool_counts": count_delta}


def route_llm(state: ChatState) -> str:
    """llm_node 的出口决策：步数用尽或没有工具调用 -> 结束"""
    if state["steps"] >= MAX_STEPS:
        return "end"
    return "tool_node" if state["messages"][-1].tool_calls else "end"  # type: ignore


def route_tool(state: ChatState) -> str:
    """tool_node 的出口决策：工具超限 -> 结束（不执行超限的那次调用）"""
    return "end" if over_limits(state) else "llm_node"


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
# 两条条件边 = 全图的所有出口，一眼看完
builder.add_conditional_edges(
    "llm_node", route_llm, {"tool_node": "tool_node", "end": END}
)
builder.add_conditional_edges(
    "tool_node", route_tool, {"llm_node": "llm_node", "end": END}
)
graph = builder.compile()
display(graph)

res = graph.invoke(
    {
        "messages": [
            SystemMessage(content="如果工具调用失败，必须根据失败结果重新制定策略"),
            HumanMessage(
                content="帮我在桌面创建一个科幻风格的网页，内容关于AI Agent。你可以先去调研一下这种网页的最佳布局再动手"
            ),
        ],
        "steps": 0,
        "tool_counts": {},
    },
    config={"recursion_limit": MAX_STEPS * 2},
)
rprint(res)

# 复现「限制退出」：把 MAX_PER_TOOL 改成 {"web_search": 1} 再执行一次，
# 第 2 次请求搜索时 route_tool 直接返回 end，不会真正发起第 2 次搜索。

## 5.1 三种「步」的计数语义：steps / tool_counts / recursion_limit



上一节案例同时用了三个限制，它们**数的是不同东西**：`steps` 是推理预算，`tool_counts` 是动作预算，`recursion_limit` 是全节点硬兜底。设限前先分清数的是哪一个「步」：



| 计数单位 | 实现 | 数什么 | 关联成本 | 一圈循环的消耗 |
|---|---|---|---|---|
| 推理轮次 | `steps`（增量 reducer）| LLM 被调用次数 | token 费用 | 每轮 +1 |
| 动作次数 | `tool_counts`（按工具名累加）| 工具被执行次数 | 工具/API 调用费用 | 每执行一组 +N |
| 节点执行 | `recursion_limit`（invoke config）| **每个节点**执行一次都 +1 | 墙钟时间 | agent + tools 一圈 **+2** |



### 为什么工具节点不计入 steps



一个正常轮次 = **1 次 LLM 推理（想）+ 可能一组工具调用（做）**，两者成本来源完全不同：



+ `steps` 数「想太久」：每步都是 token 开销，上限 ≈ 推理成本上限

+ `tool_counts` 数「做太多」：web_search 有计费、shell 有 IO 和延迟开销，与推理无关



若把 tool_node 也 +1，`MAX_STEPS = 10` 只够跑 5 轮，而且「想的次数」和「做的次数」混进一个计数器，预算语义就乱了。所以我们的案例把两者**分开设、分开查**，互不替对方省钱。



### 行业主流数的是哪一种「步」



| 方案 | 代表 | 计数方式 |
|---|---|---|
| 按轮 | Claude Code、AutoGen、Semantic Kernel | 一轮 = 1 次 LLM 生成（含期间工具调用），上限 max_iterations |
| 按节点 | LangGraph 原生 `recursion_limit` | 每个节点执行 +1，agent+tools 一圈算 2 步 |
| 按预算 | OpenAI Responses API | 步数限制 + 独立的工具/时间预算 |



主流共识是：**推理轮次和工具调用分开限制**（对应前两行），因为两者分别是 token 预算与调用预算。`recursion_limit` 属于框架兜底设计——它防的是「一切计数都失效时」的死循环，是最后一道保险，不是预算工具。



### 本案例的搭配原则



```python

res = graph.invoke(input_dict, config={"recursion_limit": MAX_STEPS * 2})  # 硬兜底

```



+ 为什么是 `MAX_STEPS * 2`：一环 = agent 节点 + tools 节点共 2 次执行，`recursion_limit` 按节点计数，这个值保证每轮 LLM 推理至少有一次完整的「想 + 做」机会，且留出一倍余量

+ 软预算失效时（比如节点里忘了更新 `steps`）→ 硬兜底拦下死循环，抛 `GraphRecursionError`，图不会无限空转

+ `recursion_limit` 的计数与图结构强耦合：节点数一变（新增检查节点、并行分支），同样的数值能跑的推理轮数就不同——这就是为什么它**不适合做预算**，只适合做兜底



一句话：**软预算（steps + tool_counts）管成本与行为，硬兜底（recursion_limit）管不死循环**，两者都设才是完整方案。


## 5.2 条件边 route vs Command：什么时候用哪个



第 5 节案例同时用了两种退出方式，它们不是竞争关系而是**分工**：



| | 条件边 route | 节点内 Command(goto=END) |
|---|---|---|
| 决策依据 | 只读 state 的纯函数 | 节点执行中的运行时信息（异常、超时、临时变量）|
| 出口可见性 | 编译期可见：`add_conditional_edges` 就是全部出口，画图/审阅清晰 | 藏在节点返回值里，要逐节点翻 |
| 职责 | 只顾决策 | 决策 + 状态更新捆绑 |
| 可测性 | route / `over_limits` 是纯函数，脱离图直接断言 | 依赖图运行环境 |
| 典型场景 | 步数/预算/终止条件等「读 state 比较」| LLM API 崩溃、工具超时、interrupt、动态跳转 |



### 为什么推荐限制类判断放条件边



+ 出口集中：全图终止路径 = 几条 `add_conditional_edges`，不用翻每个节点找命令式的 return

+ 纯函数可测：`over_limits(state)` 不依赖图，喂 state 就能断言

+ 与图语义一致：条件边是图拓扑的一部分，可视化、checkpointer 恢复都自带完整决策逻辑

+ Command 把状态更新揉进决策，同一逻辑难复用、难审查



### 什么时候 Command 才是唯一解



+ 信号不在 state 里：LLM API 抛错、工具节点内崩溃——错误发生在 route 被调用**之前**，且无法从 state 复现

+ 超时/中断：`interrupt` 之后需要从任意节点动态指定下一跳

+ 工作流模式：一次更新 + 一次跳转捆绑，多个并行分支各跳各的（Send / Command 组合）



### 本案例的分工落点



```python

def route_tool(state):

    return "end" if over_limits(state) else "llm_node"   # 限制类：条件边



def llm_node(state):

    try:

        ai_msg = model_with_tools.invoke(state["messages"])

    except Exception as e:

        return Command(goto=END)                          # 突发事件：Command

```



**写作顺序建议**：先用条件边把限制类出口画清楚，等确实出现「节点内信息才能决策」的信号时，再引入 Command。


## 5.3 复刻 opencode 的步数策略：最后一轮「收尾轮」

上一节的第 5 案例，到限时直接 END——最后一条消息可能还是带 `tool_calls` 的消息（模型吆喝了「我要搜一下」但没人替它做）。opencode 的做法是先**收尾轮**再结束，策略映射如下（v1.18.x 源码）：

| opencode 的做法 | 源码位置 | LangGraph 实现 |
|---|---|---|
| 步数 = 轮：1 次 LLM 调用（含该轮工具执行），无 super-step/节点级计数 | `packages/core/src/session/runner/llm.ts` 的 `while (needsContinuation)` | `steps` 只在 llm_node 里 +1 |
| 上限来自 agent 配置 `steps`，不设 = 不限，生产必须显式配置 | `packages/opencode/src/agent/agent.ts:54` | `MAX_STEPS` 常量显式声明 |
| 到限那一轮：不注入工具 + `toolChoice="none"` | `llm.ts:218-220` | 该轮改用**不绑定工具的模型**调用 |
| 到限那一轮：追加 `MAX_STEPS_PROMPT` 要求文本总结 | `packages/core/src/session/runner/max-steps.ts` | 追加 SystemMessage 提示总结 |
| 工具调用无独立次数预算，由权限规则管控 | agent 权限配置 | 本案例不设（需要时见 5 案例补 tool_counts）|

关键洞察：opencode 把「优雅退出」从**提示词承诺**变成了**结构保证**——最后一轮工具在接口层面就不存在（`toolChoice="none"` 且不注入工具定义），模型物理上无法再调用工具，只能输出「已完成工作」的总结文本。对比第 5 案例的「到限直接 END」，本案例保证用户拿到的**最后一条消息一定是总结文本**。


In [ ]:
# 代码案例：复刻 opencode 的步数策略 —— 最后一轮「收尾轮」优雅退出
#
# opencode 源码要点（v1.18.x）：
# 1. 步数 = 轮：一圈 = 1 次 LLM 调用（含该轮工具执行），没有 super-step/节点级计数
# 2. 上限来自 agent 配置 steps，不设 = 不限（生产必须显式配置）
# 3. 到限那一轮不收刀：不注入工具（=="toolChoice=null"）+ 追加 MAX_STEPS_PROMPT，
#    强制模型输出「已完成工作」的文本总结 -> 循环自然结束
#    （llm.ts:218-220 与 max-steps.ts）
# 4. 工具调用没有独立次数预算（opencode 用权限规则替代），本案例保持只数步数
#
# 与第 5 案例的差别：第 5 案例到限直接 END，最后一条消息可能还是带 tool_calls 的消息；
# 本案例到限先跑「收尾轮」再 END，用户拿到的最终消息一定是总结文本。

from operator import add
from typing import Annotated

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()  # 幂等：第 5 节已加载，重复调用无副作用

# ---- 复用第 5 节代码单元格已定义的 model 与 tools ----
# model 绑定工具 -> 常规轮次使用
model_with_tools = model.bind_tools(tools)

# 收尾轮专用模型：不绑定任何工具 —— 等价于 opencode 的 toolChoice="none"，
# 模型在接口层面就没有工具可调用（比提示词承诺「别调工具」更强）
model_no_tools = model

MAX_STEPS = 2

# 参考 opencode max-steps.ts 中的 MAX_STEPS_PROMPT，翻译为中文（结构保留）
MAX_STEPS_PROMPT = """关键提示 - 已达到最大步数上限

当前任务允许的最大步数已用完，工具调用已禁用，直到用户下一次输入。请仅输出文本。

严格要求：
1. 不要发起任何工具调用（不读文件、不执行命令、不联网搜索，不使用任何其他工具）
2. 必须用一条文本回复总结到目前为止已完成的工作，并结束任务"""


class SumState(MessagesState):
    steps: Annotated[int, add]   # 增量语义：节点返回 {"steps": 1}


def llm_node(state: SumState) -> dict:
    last_step = state["steps"] + 1 >= MAX_STEPS   # 下一步是最后一步 ?
    if last_step:
        # 收尾轮：不绑定工具 + 注入上限提示 -> 模型只能输出文本总结
        ai_msg = model_no_tools.invoke(
            state["messages"] + [SystemMessage(content=MAX_STEPS_PROMPT)]
        )
    else:
        ai_msg = model_with_tools.invoke(state["messages"])
    # return {"messages": [ai_msg], "steps": 1}
    return {"messages": [SystemMessage(content=MAX_STEPS_PROMPT), ai_msg], "steps": 1}


def tool_node(state: SumState) -> dict:
    last = state["messages"][-1]
    tool_map = {tool_.name: tool_ for tool_ in tools}
    tool_messages = []
    for call in last.tool_calls:
        selected = tool_map.get(call["name"])
        result = (
            f"工具调用失败: 未知工具 {call['name']}"
            if selected is None
            else selected.invoke(call["args"])
        )
        tool_messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
    return {"messages": tool_messages}


def route_llm(state: SumState) -> str:
    if state["steps"] >= MAX_STEPS:
        return "end"               # 收尾轮已产出总结文本，正常结束
    return "tool_node" if state["messages"][-1].tool_calls else "end"


builder = StateGraph(state_schema=SumState)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", route_llm, {"tool_node": "tool_node", "end": END})
builder.add_edge("tool_node", "llm_node")   # 工具执行完无条件回 LLM
graph = builder.compile()
display(graph)

res = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="帮我在桌面创建一个科幻风格的网页，内容关于AI Agent。你可以先去调研一下这种网页的最佳布局再动手"
            ),
        ],
        "steps": 0,
    },
    config={"recursion_limit": MAX_STEPS * 2},   # 硬兜底（图引擎没有 ><），参考 5.1
)

for msg in res["messages"]:
    rprint(msg.content)

# 观察点：
# 1. 把 MAX_STEPS 改小（如 2）重跑 -> 第 2 轮模型不调工具，直接输出「已完成工作」总结
# 2. 最后一条消息一定是总结文本 —— 对比第 5 案例（可能以 tool_calls 挂起句收尾）
# 3. 需要工具次数预算时：参照第 5 案例给 SumState 加 tool_counts + over_limits

